# Santander Customer Transaction Prediction — Feature Engineering Challenge

### **Goal:** Improve baseline Recall (≈0.83-0.88) through systematic feature engineering — 
### frequency encoding, row-wise stats, interactions, and outlier handling.

### **Constraints:** ONLY LOGISITIC REGRESSION. No outside data. 

### **Data:** Fetched automatically via kagglehub at runtime ,not stored in repo because the data is 268+ MB it exceeds the github limit
 



---

## KAGGLE API KEY - SETUP

## WHY KAGGLE API ?

### ---THE GIVEN DATA SET IS LARGE IT EXCEEDS THE LIMIT OF THE GITHUB STORAGE, INSTEAD OF MANUALLLY DOWNLOADING THE DATASET AND PUSHING INTO GITHUB 

### --- WE ARE SETTING UP A LOCAL KAGGLE API TOKEN FROM LOCAL **.ENV** FILE AND ACCESS USING THE **python-dotenv**

### --- I HAVE GIVEN THE **.ENV** FILE FORMAT IN THE FILE NAMED **.ENV.SAMPLE** 

## IMPLEMENTATION OF LOADING THE APIKEY FROM THE ENV

In [1]:
import os
from dotenv import load_dotenv

#loads the variable from the .env
load_dotenv()

#storing the apikey into the variable
token = os.getenv("KAGGLE_API_TOKEN")

#if the apikey is missing , we are raising a error
if not token:
    raise ValueError(
        " KAGGLE_API_TOKEN not found. "
        "Copy .env.sample to .env and add your Kaggle API token before proceeding."
    )

print(" Kaggle API token loaded successfully.")

 Kaggle API token loaded successfully.


## LOADING/IMPORTING THE DATA 

### --With the token loaded, **kagglehub** fetches the competition dataset directly into its own cache outside this repo. This avoids the 260MB+ file size limit on GitHub

### --And loading the dataset using the **Pandas** Library **pd.read_csv("path")** and storing it into the variable **df**

In [2]:
import kagglehub
import pandas as pd

# Downloads dataset to kagglehub's cache (not stored in repo)
path = kagglehub.competition_download("santander-customer-transaction-prediction")
print("Dataset downloaded to:", path)

# Confirm what's inside
import os
print("Files:", os.listdir(path))

# Load the training data
train_path = f"{path}/train.csv"
df = pd.read_csv(train_path)

print("Shape:", df.shape)
df.head()

Dataset downloaded to: /Users/pranusaravanan/.cache/kagglehub/competitions/santander-customer-transaction-prediction
Files: ['test.csv', 'train.csv', 'sample_submission.csv']
Shape: (200000, 202)


,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,train_3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,train_4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


## TRAIN_TEST_SPLIT

### WE SPLIT RANDOMLY THE DATASET INTO X_Train,X_Test,Y_Test,Y_Train

### WHERE X_Train,X_Test ---> INPUT FEATURES

### WHERE Y_Train,X_Test ---> OUTPUT FEATURES

### --AFTER LOADING THE DATA , BEFORE ANY EDA WE DO THE TRAIN_TEST_SPLIT INORDER TO REDUCE DATALEAKAGE

### --IF WE DO FEATURE ENGNEERING BEFORE THE TRAIN_TEST_SPLIT , WE ACCIDENTLY LET THE INFORMATION FROM THE DATA THAT WE USE FOR THE TEST LATER

### IMPLEMENTATION

In [3]:
from sklearn.model_selection import train_test_split

# Separate features and target first
X = df.drop(columns=["target", "ID_code"])  # drop target + ID column
y = df["target"]

#SPLITTING THE DATA INTO 85% TRAIN , 15% TEST 
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

#SPLITTING THE 85% TRAIN TO 70% TRAIN , 15% VALIDATION
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, stratify=y_train_val, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (144500, 200) (144500,)
Val: (25500, 200) (25500,)
Test: (30000, 200) (30000,)


## EXPERIMENT LOGGING

### --INSTEAD OF LOGGING THE VALUES INTO THE CSV FILE MANUALLY , WE ARE DEFINING A FUNCTION **LOG_EXPREMENT** WHICH AUTOMATICALLY UPDATES INTO CSV ONCE IT CALLS

In [4]:
import csv
import os

log_path = "../logs/experiment_log.csv"

def log_experiment(exp_id, description, recall, precision=None, f1=None, notes=""):
    file_exists = os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["experiment", "description", "recall", "precision", "f1", "notes"])
        writer.writerow([exp_id, description, recall, precision, f1, notes])

## BASELINE MODEL

### WITHOUT ANY FEATURE ENGNEERING PURE LOGISTIC REGRESSION GIVE'S US RECALL=0.77 (APPROX)

### TRAINING A LOGISTIC REGRESSION MODEL WITHOUT ANY FEATURE ENGNEERING

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train baseline Logistic Regression model
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

baseline_model.fit(X_train_scaled, y_train)

# Predictions
y_val_pred = baseline_model.predict(X_val_scaled)

# Metrics
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

# Results
print(f"Baseline Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score: {f1:.4f}")

Baseline Recall: 0.7713
Precision: 0.2837
F1 Score: 0.4149


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

## EXPLORATORY DATA ANALYSIS (EDA)

### --WE DO THIS EDA ONLY ON X_train, y_train — NEVER ON X_val OR X_test

### --IF WE LOOK AT VAL/TEST DATA EVEN JUST TO "EXPLORE", WE QUIETLY LEAK INFORMATION AND DEFEAT THE PURPOSE OF LOCKING IT AWAY

## STEP 0 - Check Dataset Quality

### Before exploring the data further, verify that the training data is clean. Check for missing values, duplicate rows, and confirm that every feature has the expected numeric data type. These quick sanity checks help ensure that later feature engineering and model training are built on reliable data.

### In this dataset, no missing values or duplicate rows are expected, but performing these checks is still a standard part of the EDA process.

In [6]:
# Missing values
print("Missing values per column:")
print(X_train.isnull().sum().sort_values(ascending=False).head())

print(f"\nTotal missing values: {X_train.isnull().sum().sum()}")

# Duplicate rows
print(f"\nDuplicate rows: {X_train.duplicated().sum()}")

# Data types
print("\nFeature data types:")
print(X_train.dtypes.value_counts())

Missing values per column:
var_0      0
var_137    0
var_127    0
var_128    0
var_129    0
dtype: int64

Total missing values: 0

Duplicate rows: 0

Feature data types:
float64    200
Name: count, dtype: int64



### --No missing values or duplicate rows were found, and all predictor columns are numeric (float64). Since the training data is already clean, no preprocessing is required before continuing with the exploratory analysis.

## STEP 1 — CHECK CLASS BALANCE

In [7]:
# checking how balanced the two target classes are
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

target
0    129979
1     14521
Name: count, dtype: int64
target
0    0.899509
1    0.100491
Name: proportion, dtype: float64


### --CLASS IMBALANCE CONFIRMED: ~90% CLASS 0, ~10% CLASS 1, THIS IS WHY RECALL AND class_weight='balanced' MATTER MORE THAN ACCURACY HERE

## STEP 2 — GLANCE AT A FEW INDIVIDUAL COLUMNS

In [8]:
# quick statistical summary of a few sample columns
X_train[["var_0", "var_1", "var_81", "var_139"]].describe()

,var_0,var_1,var_81,var_139
count,144500.000000,144500.000000,144500.000000,144500.000000
mean,10.685524,-1.635552,14.716036,7.765100
std,3.036229,4.051899,2.301991,7.692048
min,0.408400,-14.091000,7.997200,-21.274300
25%,8.460075,-4.751025,13.210200,2.387125
50%,10.528400,-1.621800,14.843500,8.072000
75%,12.767600,1.346700,16.337300,13.244300
max,20.315000,10.376800,23.132400,35.552700


### --INDIVIDUAL COLUMN CORRELATIONS ARE VERY WEAK (STRONGEST IS ~0.08), NO SINGLE COLUMN IS A STRONG PREDICTOR ON ITS OWN

## STEP 3 — RANK ALL COLUMNS BY CORRELATION WITH TARGET

### --WE TEMPORARILY COMBINE X_train AND y_train JUST TO CALCULATE CORRELATION, THIS IS STILL SAFE BECAUSE WE ONLY USE THE TRAIN SET

In [9]:
# combining X_train and y_train temporarily to compute correlation
train_corr = X_train.copy()
train_corr["target"] = y_train

# ranking every column by how strongly it relates to the target
correlations = train_corr.corr()["target"].drop("target").abs().sort_values(ascending=False)

correlations.head(20)

var_81     0.081649
var_139    0.075993
var_12     0.069859
var_6      0.065204
var_110    0.065015
var_146    0.064288
var_76     0.064026
var_174    0.061602
var_53     0.061149
var_26     0.060981
var_22     0.060191
var_21     0.059387
var_80     0.058054
var_190    0.057258
var_166    0.056907
var_99     0.056386
var_148    0.055454
var_2      0.055358
var_13     0.054843
var_165    0.054690
Name: target, dtype: float64

### --THIS IS EXACTLY WHY FEATURE ENGINEERING MATTERS FOR THIS DATASET, RAW COLUMNS ALONE DON'T CARRY MUCH SIGNAL

## STEP 4 — SAVE THE SHORTLIST

### --WE KEEP THE TOP 20 STRONGEST COLUMNS, WE REUSE THIS EXACT LIST  FOR INTERACTION FEATURES"

In [10]:
top_features = correlations.head(20).index.tolist()
print("Top 20 correlated features:", top_features)

Top 20 correlated features: ['var_81', 'var_139', 'var_12', 'var_6', 'var_110', 'var_146', 'var_76', 'var_174', 'var_53', 'var_26', 'var_22', 'var_21', 'var_80', 'var_190', 'var_166', 'var_99', 'var_148', 'var_2', 'var_13', 'var_165']


## FEATURE ENGINEERING — FREQUENCY ENCODING

### --FOR EACH COLUMN, WE ADD A NEW COLUMN THAT SAYS HOW COMMON THAT PARTICULAR VALUE IS, THIS IS USUALLY THE SINGLE BIGGEST IMPROVEMENT ON THIS DATASET

### --THE MOST IMPORTANT RULE HERE: WE CALCULATE THE FREQUENCY MAP USING ONLY X_train, THEN APPLY THE SAME MAP TO X_val AND X_test

### --IF WE CALCULATE FREQUENCIES USING ALL DATA MIXED TOGETHER, OUR SCORE WILL LOOK BETTER THAN IT ACTUALLY IS AND WON'T HOLD UP LATER WHILE VALIDATION

### IMPLEMENTATION

In [11]:
# building frequency encoding, one new column per original column
def add_frequency_encoding(df, freq_maps=None, is_train=False):
    df = df.copy()
    
    if is_train:
        freq_maps = {}
    
    for col in df.columns:
        if is_train:
            # learning the frequency map only from this (train) data
            freq_maps[col] = df[col].value_counts(normalize=True)
        
        # mapping each value to how common it is, unseen values get 0
        df[col + "_freq"] = df[col].map(freq_maps[col]).fillna(0)
    
    return df, freq_maps

# fitting the frequency map on X_train only
X_train_freq, freq_maps = add_frequency_encoding(X_train, is_train=True)

# applying the same train-derived map to val and test, not refitting
X_val_freq, _ = add_frequency_encoding(X_val, freq_maps=freq_maps, is_train=False)
X_test_freq, _ = add_frequency_encoding(X_test, freq_maps=freq_maps, is_train=False)

print("Train shape after freq encoding:", X_train_freq.shape)
print("Val shape after freq encoding:", X_val_freq.shape)

/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_9674/343190651.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col + "_freq"] = df[col].map(freq_maps[col]).fillna(0)
/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykern

Train shape after freq encoding: (144500, 400)
Val shape after freq encoding: (25500, 400)


/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_9674/343190651.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col + "_freq"] = df[col].map(freq_maps[col]).fillna(0)
/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykern

### RETRAIN THE MODEL AND CHECK SCORE

In [12]:
# scaling the new feature set
scaler_freq = StandardScaler()
X_train_freq_scaled = scaler_freq.fit_transform(X_train_freq)
X_val_freq_scaled = scaler_freq.transform(X_val_freq)

# retraining logistic regression on train + frequency features
freq_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
freq_model.fit(X_train_freq_scaled, y_train)

y_val_pred_freq = freq_model.predict(X_val_freq_scaled)

recall_freq = recall_score(y_val, y_val_pred_freq)
precision_freq = precision_score(y_val, y_val_pred_freq)
f1_freq = f1_score(y_val, y_val_pred_freq)

print("Recall after frequency encoding:", recall_freq)
print("Precision:", precision_freq, "F1:", f1_freq)

Recall after frequency encoding: 0.9863387978142076
Precision: 0.1256089074460682 F1: 0.2228395061728395


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

### LOG THIS EXPERIMENT

In [13]:
log_experiment(
    exp_id="exp_02_freq_encoding",
    description="Added frequency encoding for all 200 columns",
    recall=recall_freq,
    precision=precision_freq,
    f1=f1_freq,
    notes="Baseline was 0.7712 recall, checking improvement from freq features"
)

### --RECALL JUMPED TO 0.9863 BUT PRECISION COLLAPSED TO 0.1256 AND F1 DROPPED FROM 0.4148 TO 0.2228

### --THIS IS NOT A REAL IMPROVEMENT, THE MODEL IS LIKELY JUST PREDICTING "1" FOR MOST ROWS, HIGH RECALL ALONE IS MEANINGLESS IF PRECISION FALLS THIS HARD

### --WE NEED TO CHECK PREDICTED VS ACTUAL CLASS DISTRIBUTION TO SEE IF THE MODEL IS GENUINELY LEARNING OR JUST PREDICTING "1" MOST OF THE TIME TO GAME RECALL

In [14]:
# checking what the model is actually predicting
import numpy as np
print("Predicted class distribution:", np.bincount(y_val_pred_freq))
print("Actual class distribution:", np.bincount(y_val))


Predicted class distribution: [ 5382 20118]
Actual class distribution: [22938  2562]


### --IT IS CONFIRMED THAT PREDICTED [5382, 20118] BUT ACTUAL IS [22938, 2562], MODEL IS PREDICTING CLASS 1 FOR ~79% OF ROWS WHEN ONLY ~10% ACTUALLY ARE

### --THIS MEANS THE HIGH RECALL IS FAKE, THE MODEL ISN'T LEARNING REAL SIGNAL, IT'S JUST OVER-PREDICTING CLASS 1, LIKELY BECAUSE class_weight='balanced' PUSHED THE DECISION BOUNDARY TOO FAR ALONGSIDE THE NEW FREQUENCY FEATURES

## RETRY FREQUENCY ENCODING WITHOUT class_weight='balanced'

In [15]:
# retraining frequency-encoded features, this time without class_weight='balanced'
freq_model_v2 = LogisticRegression(max_iter=1000, random_state=42)
freq_model_v2.fit(X_train_freq_scaled, y_train)

y_val_pred_freq_v2 = freq_model_v2.predict(X_val_freq_scaled)

recall_freq_v2 = recall_score(y_val, y_val_pred_freq_v2)
precision_freq_v2 = precision_score(y_val, y_val_pred_freq_v2)
f1_freq_v2 = f1_score(y_val, y_val_pred_freq_v2)

print("Recall (no class_weight):", recall_freq_v2)
print("Precision:", precision_freq_v2, "F1:", f1_freq_v2)
print("Predicted class distribution:", np.bincount(y_val_pred_freq_v2))
log_experiment(exp_id="exp_03_freq_no_classweight",description="Frequency encoding without class_weight='balanced'",recall=recall_freq_v2,precision=precision_freq_v2,f1=f1_freq_v2,
    notes="Fixed degenerate model from exp_02. F1 still slightly below baseline (0.3977 vs 0.4148). class_weight='balanced' was the real cause of the earlier fake recall, not the freq features themselves."
)

Recall (no class_weight): 0.8040593286494926
Precision: 0.2641703000769428 F1: 0.39768339768339767
Predicted class distribution: [17702  7798]


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

### RECALL: 0.8041 | PRECISION: 0.2642 | F1: 0.3977

### --REMOVING class_weight='balanced' FIXED IT , PRECISION RECOVERED TO 0.2641, PREDICTED CLASS 1 COUNT DROPPED FROM 20118 TO 7798 

### --HOWEVER F1 (0.3977) IS STILL SLIGHTLY BELOW BASELINE (0.4148), SO FREQUENCY ENCODING ALONE ISN'T YET A CLEAR IMPROVEMENT, THE EARLIER "FAKE" RECALL WAS PURELY FROM class_weight='balanced' DISTORTING THE DECISION BOUNDARY

### --BELOW BASELINE (0.4148), FREQ ENCODING ALONE ISN'T A CLEAR WIN, BUT CONFIRMED class_weight='balanced' WAS THE REAL CAUSE OF THE EARLIER FAKE RECALL, NOT THE FREQ FEATURES

## FEATURE ENGINEERING — ROW-WISE STATISTICAL AGGREGATES

### --FOR EACH CUSTOMER (ROW), WE CALCULATE MEAN, STD, MIN, MAX, AND SKEW ACROSS ALL 200 var_ COLUMNS

### --THIS GIVES THE MODEL A "SUMMARY VIEW" OF EACH CUSTOMER THAT INDIVIDUAL COLUMNS CAN'T CAPTURE ALONE

### --NO LEAKAGE RISK HERE, THESE ARE ROW-WISE CALCULATIONS NOT LEARNED FROM TRAIN, SAME FUNCTION APPLIES SAFELY TO TRAIN/VAL/TEST

### IMPLEMENTATION

In [16]:
# building row-wise statistical aggregates across all original var_ columns
from scipy.stats import skew

def add_row_stats(df, var_cols):
    df = df.copy()
    
    # mean, std, min, max, skew computed per row across original columns only
    df["row_mean"] = df[var_cols].mean(axis=1)
    df["row_std"] = df[var_cols].std(axis=1)
    df["row_min"] = df[var_cols].min(axis=1)
    df["row_max"] = df[var_cols].max(axis=1)
    df["row_skew"] = df[var_cols].apply(lambda row: skew(row), axis=1)
    
    return df

# original var_ columns only, not the _freq columns we already added
var_cols = X_train.columns.tolist()

X_train_stats = add_row_stats(X_train_freq, var_cols)
X_val_stats = add_row_stats(X_val_freq, var_cols)
X_test_stats = add_row_stats(X_test_freq, var_cols)

print("Train shape after row stats:", X_train_stats.shape)
print("Val shape after row stats:", X_val_stats.shape)

Train shape after row stats: (144500, 405)
Val shape after row stats: (25500, 405)


In [17]:
# scaling the new feature set (freq + row stats)
scaler_stats = StandardScaler()
X_train_stats_scaled = scaler_stats.fit_transform(X_train_stats)
X_val_stats_scaled = scaler_stats.transform(X_val_stats)

# retraining logistic regression on train + freq + row stats
stats_model = LogisticRegression(max_iter=1000, random_state=42)
stats_model.fit(X_train_stats_scaled, y_train)

y_val_pred_stats = stats_model.predict(X_val_stats_scaled)

recall_stats = recall_score(y_val, y_val_pred_stats)
precision_stats = precision_score(y_val, y_val_pred_stats)
f1_stats = f1_score(y_val, y_val_pred_stats)

print("Recall after row stats:", recall_stats)
print("Precision:", precision_stats, "F1:", f1_stats)

log_experiment(
    exp_id="exp_04_row_stats",
    description="Added row-wise mean/std/min/max/skew on top of freq encoding",
    recall=recall_stats,
    precision=precision_stats,
    f1=f1_stats,
    notes="Building on exp_03 (freq encoding, no class_weight). Baseline F1=0.4148, exp_03 F1=0.3977, checking if row stats push past baseline."
)

/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

Recall after row stats: 0.6733021077283372
Precision: 0.38290788013318533 F1: 0.48818451959813214


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### RECALL: 0.6733 | PRECISION: 0.3829 | F1: 0.4882

### --FIRST STAGE TO BEAT BASELINE, ROW STATS ADDED REAL SIGNAL ON TOP OF FREQ ENCODING

## FEATURE ENGINEERING — INTERACTION FEATURES

### --WE MULTIPLY PAIRS OF OUR TOP 20 STRONGEST COLUMNS (FROM THE STAGE 2.3 SHORTLIST) TO CAPTURE COMBINED EFFECTS

### --ONLY THE SHORTLIST, NOT ALL 200 COLUMNS, PAIRWISE ACROSS EVERYTHING CREATES TOO MANY COMBINATIONS AND MOSTLY ADDS NOISE

### --SAME RULE AS BEFORE, THESE ARE COMPUTED DIRECTLY FROM EXISTING COLUMN VALUES SO THERE'S NO FITTING INVOLVED, SAFE TO APPLY IDENTICALLY TO TRAIN/VAL/TEST

### IMPLEMENTATION

In [18]:
# building pairwise interaction features from the top 20 correlated columns
from itertools import combinations

def add_interactions(df, top_cols):
    df = df.copy()
    
    # multiplying every unique pair from the shortlist
    for col1, col2 in combinations(top_cols, 2):
        df[f"{col1}_x_{col2}"] = df[col1] * df[col2]
    
    return df

# top_features was saved earlier in Stage 2.3 (top 20 by correlation)
X_train_inter = add_interactions(X_train_stats, top_features)
X_val_inter = add_interactions(X_val_stats, top_features)
X_test_inter = add_interactions(X_test_stats, top_features)

print("Train shape after interactions:", X_train_inter.shape)
print("Val shape after interactions:", X_val_inter.shape)

/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_9674/3764258917.py:9: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[f"{col1}_x_{col2}"] = df[col1] * df[col2]
/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_9674/37642

Train shape after interactions: (144500, 595)
Val shape after interactions: (25500, 595)


/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_9674/3764258917.py:9: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[f"{col1}_x_{col2}"] = df[col1] * df[col2]
/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_9674/37642

### RETRAIN THE MODEL AND CHECK SCORE

In [19]:
# scaling the new feature set (freq + row stats + interactions)
scaler_inter = StandardScaler()
X_train_inter_scaled = scaler_inter.fit_transform(X_train_inter)
X_val_inter_scaled = scaler_inter.transform(X_val_inter)

# retraining logistic regression on train + freq + row stats + interactions
inter_model = LogisticRegression(max_iter=1000, random_state=42)
inter_model.fit(X_train_inter_scaled, y_train)

y_val_pred_inter = inter_model.predict(X_val_inter_scaled)

recall_inter = recall_score(y_val, y_val_pred_inter)
precision_inter = precision_score(y_val, y_val_pred_inter)
f1_inter = f1_score(y_val, y_val_pred_inter)

print("Recall after interactions:", recall_inter)
print("Precision:", precision_inter, "F1:", f1_inter)

log_experiment(
    exp_id="exp_05_interactions",
    description="Added pairwise interaction features from top 20 correlated columns, on top of freq encoding + row stats",
    recall=recall_inter,
    precision=precision_inter,
    f1=f1_inter,
    notes="Building on exp_04 (freq + row stats, F1=0.4882, current best). Checking if interactions push F1 higher."
)

/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

Recall after interactions: 0.6807181889149102
Precision: 0.38490399470315606 F1: 0.4917524319751868


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### RECALL: 0.6807 | PRECISION: 0.3849 | F1: 0.4917

### --NEW BEST SCORE, INTERACTIONS FROM TOP 20 SHORTLIST GAVE A SMALL FURTHER LIFT OVER exp_04

## REMOVE REDUNDANT FEATURES

### --BY THIS POINT WE'VE GONE FROM 200 COLUMNS TO 400+, WE CHECK FOR COLUMNS THAT BARELY CHANGE AT ALL, AND PAIRS OF COLUMNS THAT ARE ALMOST IDENTICAL, THEN REMOVE THE REDUNDANT ONES

### --TOO MANY OVERLAPPING FEATURES MAKE THE MODEL UNSTABLE AND SLOW DOWN EVERY STEP THAT FOLLOWS

### --THRESHOLDS ARE CALCULATED ONLY ON X_train, THEN THE SAME COLUMNS TO DROP ARE APPLIED TO X_val AND X_test

### IMPLEMENTATION

In [20]:
#
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score

# --- rebuild interactions fresh, guaranteed correct, no dependency on earlier cell state ---
def add_interactions(df, top_cols):
    df = df.copy()
    new_cols = {
        f"{c1}_x_{c2}": df[c1].to_numpy() * df[c2].to_numpy()
        for c1, c2 in combinations(top_cols, 2)
    }
    interactions_df = pd.DataFrame(new_cols, index=df.index)
    return pd.concat([df, interactions_df], axis=1)

X_train_inter = add_interactions(X_train_stats, top_features)
X_val_inter = add_interactions(X_val_stats, top_features)
X_test_inter = add_interactions(X_test_stats, top_features)

# --- sanity check, will tell us immediately if source data itself is bad ---
assert X_train_inter.shape[1] > 0, "X_train_stats or top_features is empty/broken upstream"
assert not X_train_inter.isna().any().any(), "NaNs present in X_train_inter — fix upstream stats step"
n_constant = (X_train_inter.nunique() <= 1).sum()
print(f"Sanity check — shape: {X_train_inter.shape}, constant cols: {n_constant}")

# --- drop near-constant columns using nunique (avoids VarianceThreshold's crash-on-empty-result behavior) ---
nunique_train = X_train_inter.nunique()
low_variance_cols = nunique_train[nunique_train <= 1].index.tolist()
print("Near-constant columns found:", len(low_variance_cols))

# --- drop one column from each highly correlated pair (|corr| > 0.95), computed on train only ---
remaining = X_train_inter.drop(columns=low_variance_cols)
corr_matrix = remaining.corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
redundant_cols = [col for col in upper_triangle.columns if (upper_triangle[col] > 0.95).any()]
print("Highly correlated columns found:", len(redundant_cols))

# --- combine and apply the SAME drop list to train, val, test ---
cols_to_drop = list(set(low_variance_cols + redundant_cols))
print("Total columns to drop:", len(cols_to_drop))

X_train_final = X_train_inter.drop(columns=cols_to_drop)
X_val_final = X_val_inter.drop(columns=cols_to_drop)
X_test_final = X_test_inter.drop(columns=cols_to_drop)
print("Train shape after redundant removal:", X_train_final.shape)
print("Val shape after redundant removal:", X_val_final.shape)

# --- retrain and score ---
scaler_final = StandardScaler()
X_train_final_scaled = scaler_final.fit_transform(X_train_final)
X_val_final_scaled = scaler_final.transform(X_val_final)

final_model = LogisticRegression(max_iter=1000, random_state=42)
final_model.fit(X_train_final_scaled, y_train)

y_val_pred_final = final_model.predict(X_val_final_scaled)
recall_final = recall_score(y_val, y_val_pred_final)
precision_final = precision_score(y_val, y_val_pred_final)
f1_final = f1_score(y_val, y_val_pred_final)
print("Recall after redundant removal:", recall_final)
print("Precision:", precision_final, "F1:", f1_final)

log_experiment(
    exp_id="exp_07_redundant_removed",
    description=f"Dropped {len(cols_to_drop)} near-constant/highly-correlated columns from corrected interaction feature set",
    recall=recall_final,
    precision=precision_final,
    f1=f1_final,
    notes="Interactions rebuilt inline in this cell to avoid stale-state issues. Building on corrected exp_05b interactions."
)

Sanity check — shape: (144500, 595), constant cols: 0
Near-constant columns found: 0
Highly correlated columns found: 86
Total columns to drop: 86
Train shape after redundant removal: (144500, 509)
Val shape after redundant removal: (25500, 509)


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

Recall after redundant removal: 0.6760343481654957
Precision: 0.3844617092119867 F1: 0.4901655582283855


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### RECALL: 0.6760 | PRECISION: 0.3845 | F1: 0.4902

### --DROPPED 86 NEAR-CONSTANT/HIGHLY-CORRELATED COLUMNS (595→509), F1 HELD ESSENTIALLY FLAT vs exp_05 (Δ<0.002) — REDUNDANT FEATURES WEREN'T CARRYING UNIQUE SIGNAL

## TUNE THE MODEL'S SETTINGS

### --SYSTEMATICALLY TRYING DIFFERENT SETTINGS FOR THE MODEL: ITS STRICTNESS (C), ITS PENALTY STYLE (L1/L2), AND HOW IT HANDLES CLASS IMBALANCE (class_weight)

### --USING ONLY X_train_final AND X_val_final TO PICK THE BEST COMBINATION — X_test_final STAYS LOCKED AWAY UNTIL STAGE 2.9

### IMPLEMENTATION

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score
import itertools


rng = np.random.RandomState(42)
sample_idx = rng.choice(X_train_final.index, size=int(0.2 * len(X_train_final)), replace=False)
X_train_sub = X_train_final.loc[sample_idx]
y_train_sub = y_train.loc[sample_idx]

scaler_tune = StandardScaler()
X_train_sub_scaled = scaler_tune.fit_transform(X_train_sub)
X_val_tune_scaled = scaler_tune.transform(X_val_final)

param_grid = {
    "C": [0.01, 1, 100],
    "penalty": ["l1", "l2"],
    "class_weight": [None, "balanced"],
}
combos = list(itertools.product(param_grid["C"], param_grid["penalty"], param_grid["class_weight"]))
print(f"Trying {len(combos)} combinations on {len(X_train_sub)} subsampled rows...\n")

results = []
for idx, (C, penalty, class_weight) in enumerate(combos, 1):
    model = LogisticRegression(
        C=C, penalty=penalty, class_weight=class_weight,
        solver="liblinear", max_iter=300, random_state=42,
    )
    model.fit(X_train_sub_scaled, y_train_sub)
    y_val_pred = model.predict(X_val_tune_scaled)
    recall = recall_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)
    print(f"[{idx}/{len(combos)}] C={C}, penalty={penalty}, class_weight={class_weight} -> Recall={recall:.4f}, F1={f1:.4f}")
    results.append({"C": C, "penalty": penalty, "class_weight": class_weight,
                     "recall": recall, "precision": precision, "f1": f1})

results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
print("\nAll results:\n", results_df)

# ===== PART 2: SELECT BEST CONFIG SUBJECT TO THE ACTUAL OBJECTIVE =====
# Objective: recall >= 0.88, while maintaining the best F1 possible under that constraint.
# This is NOT the same as picking the global best F1 — a high-F1/low-recall config fails the assignment.
RECALL_TARGET = 0.88
qualifying = results_df[results_df["recall"] >= RECALL_TARGET].sort_values("f1", ascending=False)
print(f"\nConfigs meeting recall >= {RECALL_TARGET}:\n", qualifying)

if len(qualifying) == 0:
    print(f"\nWARNING: no config reached recall >= {RECALL_TARGET} on the subsample.")
    print("Falling back to the single highest-recall config found, so at least it's the closest available.")
    best = results_df.sort_values("recall", ascending=False).iloc[0]
else:
    best = qualifying.iloc[0]

print(f"\nSelected combo — C={best['C']}, penalty={best['penalty']}, class_weight={best['class_weight']}")
print(f"(subsample) Recall: {best['recall']:.4f} | Precision: {best['precision']:.4f} | F1: {best['f1']:.4f}")

# ===== PART 3: CONFIRM THE WINNER ON FULL TRAIN DATA =====
scaler_final2 = StandardScaler()
X_train_final_scaled = scaler_final2.fit_transform(X_train_final)
X_val_final_scaled = scaler_final2.transform(X_val_final)

final_tuned_model = LogisticRegression(
    C=best["C"], penalty=best["penalty"], class_weight=best["class_weight"],
    solver="liblinear", max_iter=1000, random_state=42,
)
final_tuned_model.fit(X_train_final_scaled, y_train)
y_val_pred_final = final_tuned_model.predict(X_val_final_scaled)

recall_tuned = recall_score(y_val, y_val_pred_final)
precision_tuned = precision_score(y_val, y_val_pred_final)
f1_tuned = f1_score(y_val, y_val_pred_final)

print(f"\nFull-data confirmation — Recall: {recall_tuned:.4f} | Precision: {precision_tuned:.4f} | F1: {f1_tuned:.4f}")

if recall_tuned >= RECALL_TARGET:
    print(f" Meets the assignment's recall >= {RECALL_TARGET} objective.")
else:
    print(f" Full-data recall ({recall_tuned:.4f}) fell short of {RECALL_TARGET} despite meeting it on the subsample — "
          f"subsample estimate didn't fully transfer. Consider widening class_weight or trying Stage 2.8's cutoff adjustment.")


log_experiment(
    exp_id="exp_08_tuned",
    description=(
        f"Grid searched on 20% subsample (12 configs), selected by filtering to recall>={RECALL_TARGET} "
        f"(assignment requirement) then maximizing F1 among qualifying configs — not global best F1. "
        f"Best: C={best['C']}, penalty={best['penalty']}, class_weight={best['class_weight']}. "
        f"Confirmed on full train set."
    ),
    recall=recall_tuned,
    precision=precision_tuned,
    f1=f1_tuned,
    notes=(
        f"Building on exp_07 (redundant removed, F1=0.4902, recall=0.6760 — below the 0.88 target). "
        f"Unconstrained best-F1 config only reached recall=0.6683, failing the objective; "
        f"this run explicitly optimizes F1 subject to recall>={RECALL_TARGET}."
    )
)

Trying 12 combinations on 28900 subsampled rows...



/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[1/12] C=0.01, penalty=l1, class_weight=None -> Recall=0.1589, F1=0.2634


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[2/12] C=0.01, penalty=l1, class_weight=balanced -> Recall=0.9219, F1=0.2919


...

/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[11/12] C=100, penalty=l2, class_weight=None -> Recall=0.6671, F1=0.4749


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


[12/12] C=100, penalty=l2, class_weight=balanced -> Recall=0.9325, F1=0.2794

All results:
          C penalty class_weight    recall  precision        f1
0     1.00      l1         None  0.667838   0.371633  0.477533
1   100.00      l1         None  0.667057   0.368716  0.474920
2   100.00      l2         None  0.667057   0.368716  0.474920
3     1.00      l2         None  0.668228   0.367540  0.474238
4     0.01      l2         None  0.711163   0.338788  0.458942
5     0.01      l1     balanced  0.921936   0.173421  0.291929
6   100.00      l2     balanced  0.932475   0.164317  0.279399
7   100.00      l1     balanced  0.932475   0.164305  0.279383
8     1.00      l2     balanced  0.932084   0.163955  0.278858
9     1.00      l1     balanced  0.932475   0.163888  0.278779
10    0.01      l1         None  0.158860   0.770833  0.263430
11    0.01      l2     balanced  0.950039   0.151039  0.260641

Configs meeting recall >= 0.88:
          C penalty class_weight    recall  precision   

/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## STAGE 2.7 EXTENDED — CUSTOM CLASS WEIGHTS + THRESHOLD TUNING

### --STAGE 2.7's grid only tried class_weight=None or "balanced" — a binary switch. "balanced" overshoots
### recall (0.92-0.95) far past the 0.88 requirement, paying an unnecessary precision cost.
### Custom weight dicts (e.g. {0:1, 1:3}) act as a DIAL instead of a switch, letting us find a weighting
### that clears recall=0.88 without overshooting it.
### --We also sweep the decision threshold on top of each weighting, since moving the cutoff is a
### separate, complementary lever to reweighting the loss function.
### --Still evaluated on X_train_final/X_val_final only — X_test_final stays locked away.

### IMPLEMENTATION

In [25]:
# STAGE 2.7 EXTENDED — custom class_weight dial + threshold sweep, scored on val only
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score

RECALL_TARGET = 0.88

scaler_cw = StandardScaler()
X_train_cw_scaled = scaler_cw.fit_transform(X_train_final)
X_val_cw_scaled = scaler_cw.transform(X_val_final)

# custom weight ratios — a dial from mild to aggressive, using your best Stage 2.7 config
# (C=1, penalty=l2) as the base, since class_weight was the dominant factor, not C/penalty
weight_ratios = [1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9]
thresholds = np.arange(0.10, 0.51, 0.02)  # sweep cutoff from 0.10 to 0.50

results = []
print(f"Sweeping {len(weight_ratios)} class weights x {len(thresholds)} thresholds...\n")

for ratio in weight_ratios:
    model = LogisticRegression(
        C=1.0,
        penalty="l2",
        class_weight={0: 1, 1: ratio},
        solver="liblinear",
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_train_cw_scaled, y_train)
    val_probs = model.predict_proba(X_val_cw_scaled)[:, 1]

    for thresh in thresholds:
        y_val_pred = (val_probs >= thresh).astype(int)
        recall = recall_score(y_val, y_val_pred)
        precision = precision_score(y_val, y_val_pred, zero_division=0)
        f1 = f1_score(y_val, y_val_pred, zero_division=0)
        results.append({
            "weight_ratio": ratio, "threshold": round(thresh, 2),
            "recall": recall, "precision": precision, "f1": f1,
        })

    print(f"weight_ratio={ratio} done — best F1 at this weight: "
          f"{max(r['f1'] for r in results if r['weight_ratio']==ratio):.4f}")

results_df = pd.DataFrame(results)

# filter to configs that meet the actual assignment requirement, then maximize F1 among those
qualifying = results_df[results_df["recall"] >= RECALL_TARGET].sort_values("f1", ascending=False)
print(f"\n{len(qualifying)} configs meet recall >= {RECALL_TARGET}")
print(qualifying.head(10))

if len(qualifying) == 0:
    print(f"\nWARNING: no config reached recall >= {RECALL_TARGET}. Widen weight_ratios or thresholds.")
    best = results_df.sort_values("recall", ascending=False).iloc[0]
else:
    best = qualifying.iloc[0]

print(f"\nBest qualifying combo — weight_ratio={best['weight_ratio']}, threshold={best['threshold']}")
print(f"Recall: {best['recall']:.4f} | Precision: {best['precision']:.4f} | F1: {best['f1']:.4f}")

# confirm the winner cleanly with its own model + threshold
final_cw_model = LogisticRegression(
    C=1.0, penalty="l2", class_weight={0: 1, 1: best["weight_ratio"]},
    solver="liblinear", max_iter=1000, random_state=42,
)
final_cw_model.fit(X_train_cw_scaled, y_train)
final_val_probs = final_cw_model.predict_proba(X_val_cw_scaled)[:, 1]
y_val_pred_final_cw = (final_val_probs >= best["threshold"]).astype(int)

recall_cw = recall_score(y_val, y_val_pred_final_cw)
precision_cw = precision_score(y_val, y_val_pred_final_cw)
f1_cw = f1_score(y_val, y_val_pred_final_cw)
print(f"\nConfirmed — Recall: {recall_cw:.4f} | Precision: {precision_cw:.4f} | F1: {f1_cw:.4f}")

log_experiment(
    exp_id="exp_09_custom_weight_threshold",
    description=(
        f"Swept custom class_weight={{0:1, 1:ratio}} for ratio in {weight_ratios} combined with "
        f"decision threshold in [0.10, 0.50]. Selected by filtering recall>={RECALL_TARGET} then "
        f"maximizing F1. Best: weight_ratio={best['weight_ratio']}, threshold={best['threshold']}."
    ),
    recall=recall_cw,
    precision=precision_cw,
    f1=f1_cw,
    notes=(
        f"Building on exp_08 (class_weight='balanced', F1=0.2919, recall=0.9219 — overshot target). "
        f"Custom weight dial + threshold search found a less aggressive setting that still clears "
        f"recall>={RECALL_TARGET} with better precision/F1."
    )
)

Sweeping 13 class weights x 21 thresholds...



/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


weight_ratio=1 done — best F1 at this weight: 0.4907


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


weight_ratio=1.5 done — best F1 at this weight: 0.4480


...

/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


weight_ratio=8 done — best F1 at this weight: 0.2764


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


weight_ratio=9 done — best F1 at this weight: 0.2679

230 configs meet recall >= 0.88
     weight_ratio  threshold    recall  precision        f1
78            2.5       0.40  0.883685   0.213867  0.344387
124           3.5       0.48  0.885636   0.213292  0.343788
101           3.0       0.44  0.886027   0.212726  0.343082
30            1.5       0.28  0.887588   0.211890  0.342109
54            2.0       0.34  0.887588   0.211260  0.341288
5             1.0       0.20  0.891881   0.208924  0.338544
146           4.0       0.50  0.890710   0.208840  0.338350
77            2.5       0.38  0.892662   0.207193  0.336324
123           3.5       0.46  0.892662   0.206315  0.335165
100           3.0       0.42  0.894223   0.205711  0.334477

Best qualifying combo — weight_ratio=2.5, threshold=0.4
Recall: 0.8837 | Precision: 0.2139 | F1: 0.3444

Confirmed — Recall: 0.8837 | Precision: 0.2139 | F1: 0.3444


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### RECALL: 0.8837 | PRECISION: 0.2139 | F1: 0.3444

### --CUSTOM CLASS_WEIGHT={0:1, 1:2.5} + THRESHOLD=0.40 FOUND A LESS AGGRESSIVE SETTING THAN "balanced",
### CLEARING THE RECALL>=0.88 REQUIREMENT WITH F1 IMPROVED ~18% OVER exp_08 (0.2919 → 0.3444)

## STAGE 2.9 — CONFIRM THE FINAL SCORE

### --THIS IS THE ONLY SCORE THAT TELLS US THE TRUTH ABOUT HOW WELL THE WORK GENERALIZES, EVERYTHING
### BEFORE THIS POINT WAS GUIDANCE TO GET US HERE

### --USING THE BEST COMBINATION FOUND: class_weight={0:1, 1:2.5}, threshold=0.40, ON THE FEATURE SET
### FROM STAGE 2.6 (X_train_final / X_test_final). X_test_final HAS NEVER BEEN TOUCHED UNTIL NOW.

### IMPLEMENTATION

In [26]:
# STAGE 2.9 — final confirmed score, on the locked-away test set, for the first and only time
import os
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix

BEST_WEIGHT_RATIO = 2.5
BEST_THRESHOLD = 0.40

# refit scaler and model on the full train set (same as exp_09, kept separate/explicit here
# so Stage 2.9 doesn't silently depend on variables left over from the tuning cell)
scaler_locked = StandardScaler()
X_train_locked_scaled = scaler_locked.fit_transform(X_train_final)
X_test_locked_scaled = scaler_locked.transform(X_test_final)

final_locked_model = LogisticRegression(
    C=1.0,
    penalty="l2",
    class_weight={0: 1, 1: BEST_WEIGHT_RATIO},
    solver="liblinear",
    max_iter=1000,
    random_state=42,
)
final_locked_model.fit(X_train_locked_scaled, y_train)

# score on the test set — the one portion of data that has never influenced any decision so far
test_probs = final_locked_model.predict_proba(X_test_locked_scaled)[:, 1]
y_test_pred = (test_probs >= BEST_THRESHOLD).astype(int)

recall_test = recall_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred)
f1_test = f1_score(y_test, y_test_pred)

print(f"FINAL TEST SET SCORE (never seen before this cell)")
print(f"Recall: {recall_test:.4f} | Precision: {precision_test:.4f} | F1: {f1_test:.4f}")
print(f"\nMeets recall >= 0.88 requirement: {'YES' if recall_test >= 0.88 else 'NO'}")

print("\nConfusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_test_pred))

# sanity check against validation performance — a big gap here would signal overfitting to val
print(f"\nFor comparison, validation score was: Recall=0.8837 | Precision=0.2139 | F1=0.3444")
gap = abs(recall_test - 0.8837)
print(f"Recall gap (test vs val): {gap:.4f} {'— looks consistent' if gap < 0.03 else '— notably different, worth investigating'}")

# save the final model, scaler, and threshold together — you need all three to reproduce predictions
os.makedirs("../artifacts", exist_ok=True)
joblib.dump(
    {
        "model": final_locked_model,
        "scaler": scaler_locked,
        "threshold": BEST_THRESHOLD,
        "cols_to_drop": cols_to_drop,   # needed to reproduce the exact feature set from raw interactions
        "top_features": top_features,   # needed to rebuild interactions from scratch if ever required
    },
    "../artifacts/final_model.pkl"
)
print("\nSaved final model bundle to artifacts/final_model.pkl")

log_experiment(
    exp_id="exp_10_final_test_score",
    description=(
        f"FINAL confirmed score on locked-away test set. Model: class_weight={{0:1, 1:{BEST_WEIGHT_RATIO}}}, "
        f"threshold={BEST_THRESHOLD}, on Stage 2.6 feature set (freq encoding + row stats + interactions, "
        f"redundant columns removed)."
    ),
    recall=recall_test,
    precision=precision_test,
    f1=f1_test,
    notes=(
        f"This is the ONLY test-set evaluation in the entire project. Validation score was F1=0.3444, "
        f"recall=0.8837. Test score confirms whether this generalizes."
    )
)

FINAL TEST SET SCORE (never seen before this cell)
Recall: 0.8879 | Precision: 0.2142 | F1: 0.3452

Meets recall >= 0.88 requirement: YES

Confusion matrix (rows=actual, cols=predicted):
[[17166  9819]
 [  338  2677]]

For comparison, validation score was: Recall=0.8837 | Precision=0.2139 | F1=0.3444
Recall gap (test vs val): 0.0042 — looks consistent

Saved final model bundle to artifacts/final_model.pkl


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### FINAL TEST SCORE — RECALL: 0.8879 | PRECISION: 0.2142 | F1: 0.3452

### --MEETS THE RECALL>=0.88 REQUIREMENT ON UNSEEN DATA. GAP FROM VALIDATION IS ONLY 0.0042 (0.8837 → 0.8879), 
### CONFIRMING THE TUNED MODEL GENERALIZES WELL AND WASN'T OVERFIT TO THE VALIDATION SET